# Audience-Adaptive Content Optimizer | Evaluator-Optimizer

In [9]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from typing_extensions import NotRequired
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
generator_llm = ChatOpenAI(model="gpt-4o")
evaluator_llm = ChatOpenAI(model="gpt-4o")

In [4]:
class EvalState(TypedDict):
    topic: str
    audience: str
    response: NotRequired[str]
    score: NotRequired[float]
    feedback: NotRequired[str]
    iteration: NotRequired[int]
    final_output: NotRequired[str]

In [5]:
SCORE_THRESHOLD = 8.0
MAX_ITERATIONS = 3

In [10]:
def parse_json(text: str):
    """Extract and parse JSON from LLM output, handling markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    return json.loads(cleaned)

def generate(state: EvalState) -> dict:
    if state.get("feedback"):
        prompt = (
            f"Improve your explanation based on this feedback.\n\n"
            f"Topic: {state['topic']}\n"
            f"Target audience: {state['audience']}\n"
            f"Previous version:\n{state['response']}\n"
            f"Score: {state.get('score', 0)}/10\n"
            f"Feedback: {state['feedback']}"
        )
    else:
        prompt = (
            f"Write a clear, engaging explanation of the following topic, "
            f"tailored specifically for the target audience.\n\n"
            f"Topic: {state['topic']}\n"
            f"Target audience: {state['audience']}\n\n"
            f"Adapt your vocabulary, examples, analogies, and depth to match "
            f"what this audience would understand and find engaging."
        )

    response = generator_llm.invoke(prompt)
    return {"response": response.content, "iteration": state.get("iteration", 0) + 1}

def evaluate(state: EvalState) -> dict:
    response = evaluator_llm.invoke(
        f"You are a content quality evaluator. Rate this explanation on a scale of 1-10.\n\n"
        f"Topic: {state['topic']}\n"
        f"Target audience: {state['audience']}\n"
        f"Explanation:\n{state['response']}\n\n"
        f"Evaluate on these criteria:\n"
        f"- Audience fit: Is the vocabulary, tone, and complexity appropriate for '{state['audience']}'?\n"
        f"- Clarity: Is it easy to follow with a logical flow?\n"
        f"- Accuracy: Are the facts and concepts correct?\n"
        f"- Engagement: Would this audience find it interesting?\n\n"
        f"Return JSON with 'score' (number) and 'feedback' (string with specific improvements needed)."
    )
    parsed = parse_json(response.content)
    score = float(parsed.get("score", 0))
    feedback = str(parsed.get("feedback", "No feedback provided."))
    return {"score": score, "feedback": feedback}

def should_continue(state: EvalState) -> Literal['finalize', 'generate']:
    score = state.get("score", 0)
    iteration = state.get("iteration", 0)
    if score >= SCORE_THRESHOLD or iteration >= MAX_ITERATIONS:
        return "finalize"
    return "generate"

def finalize(state: EvalState) -> dict:
    return {"final_output": state["response"]}

In [11]:
# Build graph
graph = StateGraph(EvalState)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", should_continue)
graph.add_edge("finalize", END)

optimizer = graph.compile()

In [12]:
# Plot the optimizer
plot_mermaid(optimizer)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	evaluate(evaluate)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	evaluate -.-> finalize;
	evaluate -.-> generate;
	generate --> evaluate;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [13]:
result = optimizer.invoke({
    "topic": "How blockchain works",
    "audience": "Senior executives with no technical background"
})
print(f"Final score: {result['score']}/10")
print(f"Iterations: {result['iteration']}")
print(result["final_output"])

Final score: 8.0/10
Iterations: 1
Certainly! Let's explore how blockchain works through a lens that's relevant and engaging for senior executives with no technical background.

### Understanding Blockchain: The Digital Ledger

Imagine your organization's financial ledger, but instead of it being just one book in one location, it's duplicated across thousands of computers globally. This is the essence of blockchain—it's a distributed ledger that's maintained simultaneously across a network of computers.

### Breaking Down the Basics:

1. **Blocks and Chains**:
   Think of blockchain as a chain of digital "blocks". Each block is like a page in a ledger and contains a list of transactions, much like entries on a balance sheet. These blocks are linked together chronologically and securely, forming a "chain", hence the name "blockchain".

2. **Decentralization**:
   Traditionally, transactions go through a central authority, like a bank. Blockchain reinvents this by allowing transactions to

In [14]:
stream_invoke(optimizer, {
    "topic": "How blockchain works",
    "audience": "Senior executives with no technical background"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'topic': 'How blockchain works',
 'audience': 'Senior executives with no technical background',
 'response': 'Certainly! Let\'s explore how blockchain works through a lens that\'s relevant and engaging for senior executives with no technical background.\n\n### Understanding Blockchain: The Digital Ledger\n\nImagine your organization\'s financial ledger, but instead of it being just one book in one location, it\'s duplicated across thousands of computers globally. This is the essence of blockchain—it\'s a distributed ledger that\'s maintained simultaneously across a network of computers.\n\n### Breaking Down the Basics:\n\n1. **Blocks and Chains**:\n   Think of blockchain as a chain of digital "blocks". Each block is like a page in a ledger and contains a list of transactions, much like entries on a balance sheet. These blocks are linked together chronologically and securely, forming a "chain", hence the name "blockchain".\n\n2. **Decentralization**:\n   Traditionally, transactions go 